# Boundary Fix Analysis — Four Methods, All Correcting ω AND div

BEACH omega profiles end at ~16 km / ~100 hPa with ω_top ≠ 0.
BEACH derives **ω** and **div** independently — they are not mutually consistent,
which is why the boundary term `h_top·ω_top/g ≈ ±5000 W/m²` dominates Methods 2 & 3.

All four fixes below correct **both** ω **and** div so that ∂ω_corr/∂p = −div_corr holds.

| Method | ω correction | div correction | Domain |
|---|---|---|---|
| **M1 — Hard-zero** | Step to 0 at top level | Δdiv at top level only | BEACH |
| **M2 — Linear ramp** | Linear ramp to 0 over full column | Depth-uniform Δdiv | BEACH |
| **M3 — ERA5 extension** | BEACH ω + ERA5 ω above | BEACH div + ERA5 div (−∂ω/∂p) | BEACH + ERA5 |
| **M4 — Hybrid** | Linear-ramp BEACH + ERA5 above | Ramp-corrected BEACH div + ERA5 div | BEACH + ERA5 |

## 0. Setup

In [ ]:
import sys
sys.path.insert(0, '..')
import numpy as np
import matplotlib.pyplot as plt

from scripts.mse_budget import (
    load_dataset, compute_budget,
    apply_step_zero_consistent,
    apply_mass_correction,
)
from scripts.era5_extension import stitch_beach_era5, compute_budget_ext
from scripts.config import default_era5_omega_path

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
TOP_COLOR = '#d62728'
BOT_COLOR = '#1f77b4'


In [ ]:
ds  = load_dataset()
cat = ds['category_plane'].values
mask_top = np.array(['Top-Heavy'    in str(c) for c in cat])
mask_bot = np.array(['Bottom-Heavy' in str(c) for c in cat])
p_m  = ds['p_mean'].values
omega = ds['omega'].values
print(f'Circles: {ds.sizes["circle"]}  |  Top: {mask_top.sum()}  |  Bot: {mask_bot.sum()}')


## 1. Run All Four Methods

Each method corrects both ω and div, then runs the full 3-method MSE budget.

In [ ]:
era5_path = default_era5_omega_path()

# Baseline (no correction)
budget_raw = compute_budget(ds)

# M1 — hard-zero at top level + matching div correction at that level
budget_m1 = compute_budget(apply_step_zero_consistent(ds))

# M2 — linear O'Brien ramp + depth-uniform div correction (standard fix)
ds_mc, _ = apply_mass_correction(ds)
budget_m2 = compute_budget(ds_mc)

# M3 — ERA5 extension: BEACH ω + ERA5 ω above; BEACH div + ERA5 div (-dω/dp)
ds_ext    = stitch_beach_era5(ds, era5_path)
budget_m3 = compute_budget_ext(ds_ext)

# M4 — Hybrid: linear-ramp BEACH first, then ERA5 above
ds_ext_mc = stitch_beach_era5(ds_mc, era5_path)
budget_m4 = compute_budget_ext(ds_ext_mc)

print('All budgets computed OK')


## 2. Omega Profiles — What Each Method Does to ω

Group-mean ω profiles showing the correction applied to the BEACH column.

In [ ]:
om_raw_top  = np.nanmean(omega[mask_top], axis=0)
om_raw_bot  = np.nanmean(omega[mask_bot], axis=0)

om_m1_top = np.nanmean(apply_step_zero_consistent(ds)['omega'].values[mask_top], axis=0)
om_m2_top = np.nanmean(ds_mc['omega'].values[mask_top], axis=0)
# ERA5 extended omega (stitched column, shown up to 20 hPa)
om_m3_top = np.nanmean(ds_ext['omega_ext'].values[mask_top], axis=0)
p_ext_top  = np.nanmean(ds_ext['p_ext'].values[mask_top], axis=0) / 100
p_top_disp = np.nanmean(p_m[mask_top], axis=0) / 100

fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)

# Left: BEACH-only corrections (raw vs M1 vs M2)
ok = p_top_disp < 250
axes[0].plot(om_raw_top[ok], p_top_disp[ok], 'k-',  lw=2, label='Raw')
axes[0].plot(om_m1_top[ok], p_top_disp[ok], color='orange', lw=2, ls='--', label='M1 hard-zero')
axes[0].plot(om_m2_top[ok], p_top_disp[ok], color='green',  lw=2, ls=':', label='M2 linear-ramp')
axes[0].axvline(0, color='k', lw=0.8, ls=':')
axes[0].invert_yaxis(); axes[0].set_ylim(250, 20)
axes[0].set_xlabel(r'$\omega$ (Pa s$^{-1}$)'); axes[0].set_ylabel('Pressure (hPa)')
axes[0].set_title('Top-Heavy — BEACH corrections'); axes[0].legend(fontsize=9); axes[0].grid(True, alpha=0.3)

# Centre: ERA5 extension (raw BEACH vs M3 stitched)
ok_ext = np.isfinite(p_ext_top)
axes[1].plot(om_raw_top[ok], p_top_disp[ok], 'k-', lw=2, label='BEACH only')
axes[1].plot(om_m3_top[ok_ext], p_ext_top[ok_ext], color='purple', lw=2, ls='--', label='M3 BEACH+ERA5')
axes[1].axvline(0, color='k', lw=0.8, ls=':')
axes[1].invert_yaxis(); axes[1].set_ylim(1000, 20)
axes[1].set_xlabel(r'$\omega$ (Pa s$^{-1}$)')
axes[1].set_title('Top-Heavy — ERA5 extension to 20 hPa'); axes[1].legend(fontsize=9); axes[1].grid(True, alpha=0.3)

# Right: boundary term comparison
bnd_vals = {n: b['h_div_residual'].values
            for n, b in [('Raw',budget_raw),('M1',budget_m1),
                          ('M2',budget_m2),('M3',budget_m3),('M4',budget_m4)]}
names = list(bnd_vals.keys())
top_bnd = [np.nanmean(np.abs(bnd_vals[n][mask_top])) for n in names]
bot_bnd = [np.nanmean(np.abs(bnd_vals[n][mask_bot])) for n in names]
x = np.arange(len(names))
axes[2].bar(x - 0.2, top_bnd, 0.4, color=TOP_COLOR, alpha=0.7, label='Top-Heavy')
axes[2].bar(x + 0.2, bot_bnd, 0.4, color=BOT_COLOR, alpha=0.7, label='Bottom-Heavy')
axes[2].set_xticks(x); axes[2].set_xticklabels(names)
axes[2].set_ylabel(r'$|\langle h\nabla\cdot v\rangle - \langle\omega\partial h/\partial p\rangle|$ (W m$^{-2}$)')
axes[2].set_title('Boundary term |mean|'); axes[2].legend(); axes[2].grid(True, alpha=0.3, axis='y')
axes[2].set_yscale('log')

fig.suptitle('All 4 methods — omega profiles and boundary term', fontweight='bold')
plt.tight_layout()
plt.show()


## 3. GMS Comparison

GMS_adv = `mean(vert_adv) / mean(vert_adv_dse)` — ratio of group means.

In [ ]:
all_budgets = {
    'Raw':    budget_raw,
    'M1\nhard-zero': budget_m1,
    'M2\nlinear-ramp': budget_m2,
    'M3\nERA5 ext': budget_m3,
    'M4\nhybrid': budget_m4,
}

clip = 3
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

for ax, mask, color, label in [
    (axes[0], mask_top, TOP_COLOR, 'Top-Heavy'),
    (axes[1], mask_bot, BOT_COLOR, 'Bottom-Heavy'),
]:
    gms_vals = [b['gms_adv'].values for b in all_budgets.values()]
    data   = [np.clip(v[mask][np.isfinite(v[mask])], -clip, clip) for v in gms_vals]
    means  = [np.nanmean(b['vert_adv'].values[mask]) /
               np.nanmean(b['vert_adv_dse'].values[mask]) for b in all_budgets.values()]
    pos = np.arange(len(all_budgets))
    bp  = ax.boxplot(data, positions=pos, patch_artist=True, widths=0.5,
                     showfliers=False, medianprops=dict(color='k', lw=2))
    for patch in bp['boxes']:
        patch.set_facecolor(color); patch.set_alpha(0.5)
    ax.plot(pos, means, 'k^', ms=9, zorder=5, label='group mean')
    ax.axhline(0, color='k', lw=0.8, ls='--')
    ax.set_xticks(pos); ax.set_xticklabels(list(all_budgets.keys()), fontsize=9)
    ax.set_ylabel('GMS_adv'); ax.set_title(f'{label} (n={mask.sum()})')
    ax.set_ylim(-clip, clip); ax.grid(True, alpha=0.3, axis='y')
    ax.legend(fontsize=9)

fig.suptitle('GMS_adv — all 4 boundary fixes', fontweight='bold')
plt.tight_layout()
plt.show()


## 4. Summary Table

In [ ]:
def _gms(b, m):
    va=b['vert_adv'].values[m]; vds=b['vert_adv_dse'].values[m]
    mv=np.nanmean(vds); return np.nanmean(va)/mv if abs(mv)>1 else float('nan')
def _bnd(b, m): return np.nanmean(np.abs(b['h_div_residual'].values[m]))

rows = [
    ('Raw',              budget_raw),
    ('M1  hard-zero+div',budget_m1),
    ('M2  linear-ramp+div', budget_m2),
    ('M3  ERA5+ERA5div', budget_m3),
    ('M4  hybrid (ramp→ERA5)', budget_m4),
]
hdr = f'{"Method":<28} {"TopGMS":>8} {"BotGMS":>8} {"Top|bnd|W/m²":>14} {"Bot|bnd|W/m²":>14}'
print(hdr)
print('─'*len(hdr))
for name, bud in rows:
    print(f'{name:<28} {_gms(bud,mask_top):>+8.4f} {_gms(bud,mask_bot):>+8.4f}'
          f' {_bnd(bud,mask_top):>14.1f} {_bnd(bud,mask_bot):>14.1f}')


## 5. Interpretation

### Why M1 (hard-zero) only halves the boundary term
The hard-zero correction focuses all the div adjustment at a **single pressure level** (the profile top).
The rest of the BEACH column still has the original div–ω inconsistency intact.
The boundary term is halved because we zeroed ω_top (removing the explicit `h_top·ω_top/g` term)
but the integral mismatch ∫(∂ω/∂p + div)dp throughout the column remains.

### Why M2 (linear ramp) works best among BEACH-only methods
The linear ramp is the **minimum-norm** correction: it redistributes the needed Δω
smoothly across the whole column and applies the corresponding **constant Δdiv** everywhere.
After correction ∂ω_corr/∂p = −div_corr holds exactly at every level, so the identity closes.
The residual ~30 W/m² is the irreducible estimation noise between independently-derived BEACH ω and div.

### Why M3 (ERA5 extension) doesn't close the boundary
Adding ERA5 ω and ERA5 div above 150 hPa extends the **ERA5 part consistently** (ERA5 satisfies
∂ω/∂p = −div internally). But the **BEACH part** still has its original inconsistency.
The BEACH column residual dominates the total boundary term, so M3 ≈ Raw.

### Why M4 (hybrid) is slightly worse than M2
After the linear ramp, BEACH ω_corr_top ≈ 0. The ERA5 ω at the first ERA5 level (100 hPa)
is not zero — this **jump at the BEACH–ERA5 junction** creates a new boundary term.
The combined residual (~87 W/m²) is larger than M2 alone (~30 W/m²) because of this junction mismatch.
A proper M4 would need to smoothly blend BEACH and ERA5 ω at the junction.

### Conclusion for RQ2
**Use M2 (linear ramp)** as the standard correction.  
M3/M4 confirm that the GMS signal is **entirely tropospheric** — ERA5 levels above 150 hPa
contribute < 1% to GMS_adv.  The linear-ramp residual (~30 W/m²) is the accepted irreducible error.